In [67]:
import pandas as pd
import numpy as np
from scipy.interpolate import CubicSpline, interp1d
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.signal import butter, filtfilt
import os
from scipy.signal import find_peaks

In [68]:
# =====================================================
# Parâmetros e caminhos
# =====================================================
sujeito = "03"  # Número do sujeito (ex.: "01", "02", ...)

file_path_sp = f"Input_ML/{sujeito}(sl).csv"
file_path_vicon = f"Input_ML/{sujeito}(vc).xlsx"

# Carregar dados
df_sp = pd.read_csv(file_path_sp)                    # Smartphone (acel + giro)
df_vc = pd.read_excel(file_path_vicon, skiprows=3)   # Vicon

# Parâmetros de amostragem
fs = 60                 # Frequência de amostragem (Hz)
dt = 1 / fs

In [69]:
# =====================================================
# Função auxiliar: mesma lógica de interpolação do smartphone
# (CubicSpline, cumsum, remoção de duplicados e reordenação)
# =====================================================
def interpolar_smartphone(time_array, signal_array, dt):
    Time_sp = np.asarray(time_array, dtype=float)
    Signal_sp = np.asarray(signal_array, dtype=float)

    # Remover NaNs (timestamps ou leituras ausentes)
    mask_valid = ~np.isnan(Time_sp) & ~np.isnan(Signal_sp)
    Time_sp = Time_sp[mask_valid]
    Signal_sp = Signal_sp[mask_valid]

    Time_diff_sp = np.diff(Time_sp)
    Time_diff_sp = np.insert(Time_diff_sp, 0, 0)
    Time_cumsum_sp = np.cumsum(Time_diff_sp)

    if not np.all(np.diff(Time_cumsum_sp) > 0):
        unique_indices_sp = np.unique(Time_cumsum_sp, return_index=True)[1]
        Time_cumsum_sp = Time_cumsum_sp[unique_indices_sp]
        Signal_sp = Signal_sp[unique_indices_sp]
        sorted_indices_sp = np.argsort(Time_cumsum_sp)
        Time_cumsum_sp = Time_cumsum_sp[sorted_indices_sp]
        Signal_sp = Signal_sp[sorted_indices_sp]

    New_Time_sp = np.arange(Time_cumsum_sp.min(), Time_cumsum_sp.max(), dt)
    cs_sp = CubicSpline(Time_cumsum_sp, Signal_sp)
    Signal_interpolated_sp = cs_sp(New_Time_sp)

    return Time_cumsum_sp, Signal_sp, New_Time_sp, Signal_interpolated_sp

# =====================================================
# 1. Aceleração do Smartphone - Interpolação (X, Y, Z)
# =====================================================
Time_acc_sp = df_sp["accelerometerTimestamp_sinceReboot(s)"].values

# Conversão de G -> m/s² (mesma lógica do notebook original: * -9.80665)
Signal_accX_raw = df_sp["accelerometerAccelerationX(G)"].values * -9.80665
Signal_accY_raw = df_sp["accelerometerAccelerationY(G)"].values * -9.80665
Signal_accZ_raw = df_sp["accelerometerAccelerationZ(G)"].values * -9.80665

# Remoção do componente gravitacional (apenas no eixo Y, como no notebook original)
Signal_accY_raw -= 9.80665

Time_cumsum_accX_sp, Signal_accX_sp, New_Time_accX_sp, Signal_interpolated_accX_sp = \
    interpolar_smartphone(Time_acc_sp, Signal_accX_raw, dt)
Time_cumsum_accY_sp, Signal_accY_sp, New_Time_accY_sp, Signal_interpolated_accY_sp = \
    interpolar_smartphone(Time_acc_sp, Signal_accY_raw, dt)
Time_cumsum_accZ_sp, Signal_accZ_sp, New_Time_accZ_sp, Signal_interpolated_accZ_sp = \
    interpolar_smartphone(Time_acc_sp, Signal_accZ_raw, dt)

# =====================================================
# 2. Giroscópio do Smartphone - Interpolação (X, Y, Z)
# =====================================================
Time_gyro_sp = df_sp["gyroTimestamp_sinceReboot(s)"].values

# Giroscópio já está em rad/s (sem conversão)
Signal_gyroX_raw = df_sp["gyroRotationX(rad/s)"].values
Signal_gyroY_raw = df_sp["gyroRotationY(rad/s)"].values
Signal_gyroZ_raw = df_sp["gyroRotationZ(rad/s)"].values

Time_cumsum_gyroX_sp, Signal_gyroX_sp, New_Time_gyroX_sp, Signal_interpolated_gyroX_sp = \
    interpolar_smartphone(Time_gyro_sp, Signal_gyroX_raw, dt)
Time_cumsum_gyroY_sp, Signal_gyroY_sp, New_Time_gyroY_sp, Signal_interpolated_gyroY_sp = \
    interpolar_smartphone(Time_gyro_sp, Signal_gyroY_raw, dt)
Time_cumsum_gyroZ_sp, Signal_gyroZ_sp, New_Time_gyroZ_sp, Signal_interpolated_gyroZ_sp = \
    interpolar_smartphone(Time_gyro_sp, Signal_gyroZ_raw, dt)

# =====================================================
# 3. Deslocamento do Esterno - Vicon - Interpolação (apenas Esterno_Z)
# =====================================================
df_vc.columns = ['Frame', 'Time', 'Esterno_X', 'Esterno_Y', 'Esterno_Z', 'C7_X', 'C7_Y', 'C7_Z']
df_vc['Time'] = pd.to_numeric(df_vc['Time'], errors='coerce')
df_vc['Esterno_Z'] = pd.to_numeric(df_vc['Esterno_Z'], errors='coerce')
df_vc.dropna(subset=['Time', 'Esterno_Z'], inplace=True)

Time_vc = df_vc['Time'].to_numpy()
Signal_vc = df_vc['Esterno_Z'].to_numpy()

Time_diff_vc = np.diff(Time_vc)
Time_diff_vc = np.insert(Time_diff_vc, 0, 0)
Time_cumsum_vc = np.cumsum(Time_diff_vc)

if not np.all(np.diff(Time_cumsum_vc) > 0):
    unique_indices_vc = np.unique(Time_cumsum_vc, return_index=True)[1]
    Time_cumsum_vc = Time_cumsum_vc[unique_indices_vc]
    Signal_vc = Signal_vc[unique_indices_vc]
    sorted_indices_vc = np.argsort(Time_cumsum_vc)
    Time_cumsum_vc = Time_cumsum_vc[sorted_indices_vc]
    Signal_vc = Signal_vc[sorted_indices_vc]

New_Time_vc = np.arange(Time_cumsum_vc.min(), Time_cumsum_vc.max(), dt)
cs_linear = interp1d(Time_cumsum_vc, Signal_vc, kind='linear')
Signal_interpolated_vc = cs_linear(New_Time_vc)

In [70]:
# =====================================================
# 4. Filtro passa-baixa — aceleração Y do smartphone (fc = 1 Hz)
# =====================================================
# Saída: Signal_filtered_accY_sp (mesmo eixo temporal que New_Time_accY_sp)

_fc_hz = 3.0
_fs_hz = float(fs)
_nyq = 0.5 * _fs_hz
_wn = _fc_hz / _nyq
_b, _a = butter(4, _wn, btype="low")
Signal_filtered_accY_sp = filtfilt(
    _b, _a, np.asarray(Signal_interpolated_accY_sp, dtype=float)
)

_t_accy = np.asarray(New_Time_accY_sp, dtype=float)
_y_raw = np.asarray(Signal_interpolated_accY_sp, dtype=float)
_y_filt = np.asarray(Signal_filtered_accY_sp, dtype=float)

fig_filt_accy = go.Figure()
fig_filt_accy.add_trace(go.Scatter(
    x=_t_accy, y=_y_raw, mode="lines", name="Acel. Y interpolada",
    line=dict(color="gray", width=1),
    opacity=0.55,
))
fig_filt_accy.add_trace(go.Scatter(
    x=_t_accy, y=_y_filt, mode="lines", name=f"Acel. Y filtrada (fc = {_fc_hz:g} Hz)",
    line=dict(color="deepskyblue", width=2),
))
fig_filt_accy.update_layout(
    title=f"Aceleração Y — filtro passa-baixa Butterworth 4ª ordem (fc = {_fc_hz:g} Hz, fs = {_fs_hz:g} Hz)",
    xaxis_title="Tempo (s)",
    yaxis_title="Aceleração (m/s²)",
    template="plotly_dark",
    width=1500,
    height=450,
    plot_bgcolor="black",
    paper_bgcolor="black",
    font=dict(color="white"),
    legend=dict(x=0.01, y=0.99),
)
fig_filt_accy.show()


In [71]:
# =====================================================
# 5. Detecção de Picos, Vales, Zero Crossings e Corte Automático (30.5 s)
# =====================================================


# --- Sinal de trabalho (aceleração Y filtrada) ---
sinal  = Signal_filtered_accY_sp
tempo  = New_Time_accY_sp

# --- Parâmetros ---
altura_minima         = 1  # m/s²  - limiar mínimo para aceitar um pico/vale
silencio_dur          = 1.0   # s     - duração mínima de silêncio antes do 1º pico
janela_sequencia      = 15.0  # s     - janela após o pico para validar sequência
min_picos_sequencia   = 4     # picos - nº mínimo de picos (incl. o candidato) na janela
gap_max_picos         = 3  # s     - gap máximo entre picos consecutivos na sequência
duracao_corte         = 30.5  # s     - duração a partir do ZC anterior ao 1º pico

# =====================================================
# (a) Zero crossings com interpolação linear
# =====================================================
zc_indices = np.where(np.diff(np.sign(sinal)))[0]
zc_times = []
for i in zc_indices:
    x1, x2 = tempo[i], tempo[i+1]
    y1, y2 = sinal[i], sinal[i+1]
    if (y2 - y1) != 0:
        zc_times.append(x1 - y1 * (x2 - x1) / (y2 - y1))
zc_times = np.array(zc_times)

# =====================================================
# (b) Picos e vales válidos (acima do limiar)
# =====================================================
picos_idx, _ = find_peaks( sinal, height=altura_minima)
vales_idx, _ = find_peaks(-sinal, height=altura_minima)

# =====================================================
# (c) Primeiro pico válido - critérios simultâneos:
#     (1) Silêncio contínuo imediatamente antes do pico
#         com duração >= silencio_dur  (1 s, podendo ser maior)
#         Silêncio: |sinal| < altura_minima
#     (2) Sequência consistente e REGULAR depois do pico:
#         - pelo menos min_picos_sequencia picos (incluindo o candidato)
#           dentro da janela de janela_sequencia segundos;
#         - gap entre picos consecutivos dessa sequência inicial
#           <= gap_max_picos  (rejeita pico isolado antes de platô)
# =====================================================
picos_tempos = tempo[picos_idx]

def _sequencia_regular(p_pos):
    """Verifica se, a partir do pico em picos_idx[p_pos], existem
    min_picos_sequencia picos consecutivos dentro de janela_sequencia
    e com gap máximo <= gap_max_picos."""
    if p_pos + min_picos_sequencia - 1 >= len(picos_idx):
        return False
    seq_t = tempo[picos_idx[p_pos : p_pos + min_picos_sequencia]]
    if (seq_t[-1] - seq_t[0]) > janela_sequencia:
        return False
    gaps = np.diff(seq_t)
    if np.any(gaps > gap_max_picos):
        return False
    return True

primeiro_pico = None
for p_pos, p in enumerate(picos_idx):
    t_p = tempo[p]

    # (1) Silêncio anterior
    j = p - 1
    while j >= 0 and abs(sinal[j]) < altura_minima:
        j -= 1
    dur_silencio = tempo[p] - tempo[j + 1] if (j + 1) <= p else 0.0
    if dur_silencio < silencio_dur:
        continue

    # (2) Sequência posterior regular
    if not _sequencia_regular(p_pos):
        continue

    primeiro_pico = p
    break

# Fallback: se nenhum pico passar nos dois critérios, tenta apenas
# a "sequência regular" (sem exigir silêncio prévio estrito)
if primeiro_pico is None and len(picos_idx) > 0:
    for p_pos, p in enumerate(picos_idx):
        if _sequencia_regular(p_pos):
            primeiro_pico = int(p)
            print("[aviso] Nenhum pico com silêncio >= "
                  f"{silencio_dur:.1f}s + sequência regular - "
                  "usando 1º pico com sequência regular.")
            break

if primeiro_pico is None and len(picos_idx) > 0:
    primeiro_pico = int(picos_idx[0])
    print("[aviso] Fallback final: usando o 1º pico detectado.")

t_primeiro_pico = tempo[primeiro_pico]

# =====================================================
# (d1) Âncora p/ alinhamento com Vicon: 1º zero crossing DEPOIS do 1º pico
#      (equivale, na exportação, ao 1º pico do Vicon)
# =====================================================
zc_posteriores = zc_times[zc_times > t_primeiro_pico]
if len(zc_posteriores) > 0:
    t_zc_apos_pico_sp = float(zc_posteriores[0])
else:
    t_zc_apos_pico_sp = float(tempo[primeiro_pico])
    print("[aviso] Nenhum ZC após o 1º pico; âncora de alinhamento = tempo do pico.")

# =====================================================
# (d2) Início do corte: 1º zero crossing ANTES do 1º pico; 30.5 s a partir daqui
# =====================================================
zc_anteriores = zc_times[zc_times < t_primeiro_pico]
t_inicio = float(zc_anteriores[-1]) if len(zc_anteriores) > 0 else float(tempo[0])

# =====================================================
# (e) Corte: t_inicio (ZC pré-pico) até t_inicio + 30.5 s
# =====================================================
t_fim = min(t_inicio + duracao_corte, tempo[-1])
mask_corte   = (tempo >= t_inicio) & (tempo <= t_fim)
tempo_cortado = tempo[mask_corte] - t_inicio   # zerar o tempo
sinal_cortado = sinal[mask_corte]

# Eventos dentro do intervalo do corte (em tempo zerado)
picos_cort_idx  = [i for i in picos_idx if t_inicio <= tempo[i] <= t_fim]
vales_cort_idx  = [i for i in vales_idx if t_inicio <= tempo[i] <= t_fim]
zc_cort_times   = [t for t in zc_times  if t_inicio <= t       <= t_fim]

picos_t = [tempo[i] - t_inicio for i in picos_cort_idx]
picos_y = [sinal[i]            for i in picos_cort_idx]
vales_t = [tempo[i] - t_inicio for i in vales_cort_idx]
vales_y = [sinal[i]            for i in vales_cort_idx]
zc_t    = [t - t_inicio        for t in zc_cort_times]

print(f"1º pico válido (acel. Y filtrada): t = {t_primeiro_pico:.3f} s")
print(f"1º ZC após o pico (âncora alinhamento ↔ Vicon): t = {t_zc_apos_pico_sp:.3f} s")
print(f"Início do corte (1º ZC antes do pico): t = {t_inicio:.3f} s")
print(f"Fim do corte: t = {t_fim:.3f} s  (duração = {t_fim - t_inicio:.3f} s)")

# =====================================================
# (f) Plot - Sinal filtrado cortado + picos, vales e zero crossings
# =====================================================
fig_corte = go.Figure()

fig_corte.add_trace(go.Scatter(
    x=tempo_cortado, y=sinal_cortado,
    mode='lines', name='Sinal Filtrado (Cortado)',
    line=dict(color='deepskyblue')
))

fig_corte.add_trace(go.Scatter(
    x=picos_t, y=picos_y, mode='markers', name='Picos',
    marker=dict(color='red', size=9, symbol='circle')
))

fig_corte.add_trace(go.Scatter(
    x=vales_t, y=vales_y, mode='markers', name='Vales',
    marker=dict(color='blue', size=9, symbol='circle')
))

fig_corte.add_trace(go.Scatter(
    x=zc_t, y=[0] * len(zc_t), mode='markers', name='Zero Crossings',
    marker=dict(color='yellow', size=8, symbol='circle-open')
))

fig_corte.add_vline(
    x=float(t_zc_apos_pico_sp - t_inicio),
    line_dash="dash", line_color="lime", line_width=2,
    annotation_text="ZC pós-pico (âncora Vicon)", annotation_position="top",
)

fig_corte.update_layout(
    title="Corte (30.5 s) — início no 1º ZC antes do pico; linha = ZC pós-pico (âncora)",
    xaxis_title="Tempo (s) desde o início do corte",
    yaxis_title="Aceleração (m/s²)",
    template="plotly_dark",
    width=1500, height=450,
    plot_bgcolor='black', paper_bgcolor='black', font=dict(color='white'),
    legend=dict(x=1.02, y=1, xanchor='left', yanchor='top', bordercolor='gray', borderwidth=1)
)

fig_corte.show()

[aviso] Nenhum pico com silêncio >= 1.0s + sequência regular - usando 1º pico com sequência regular.
1º pico válido (acel. Y filtrada): t = 2.367 s
1º ZC após o pico (âncora alinhamento ↔ Vicon): t = 2.459 s
Início do corte (1º ZC antes do pico): t = 2.206 s
Fim do corte: t = 32.706 s  (duração = 30.500 s)


In [72]:
# =====================================================
# 9. Smartphone - Guardar giro e acel (x,y,z) já cortados no tempo do sinal filtrado
#     (usa o t_inicio/t_fim calculado no corte automático do sinal filtrado)
# =====================================================

# Preservar os tempos do corte do smartphone (antes do bloco do Vicon)
t_inicio_sp = float(t_inicio)
t_fim_sp = float(t_fim)

# Usar como base de tempo o mesmo eixo usado no corte (accY)
t_sp = np.asarray(New_Time_accY_sp, dtype=float)

# Alinhar todos os sinais do smartphone nessa base de tempo
accX_sp = np.interp(t_sp, np.asarray(New_Time_accX_sp, dtype=float), np.asarray(Signal_interpolated_accX_sp, dtype=float))
accY_sp = np.interp(t_sp, np.asarray(New_Time_accY_sp, dtype=float), np.asarray(Signal_interpolated_accY_sp, dtype=float))
accZ_sp = np.interp(t_sp, np.asarray(New_Time_accZ_sp, dtype=float), np.asarray(Signal_interpolated_accZ_sp, dtype=float))

gyroX_sp = np.interp(t_sp, np.asarray(New_Time_gyroX_sp, dtype=float), np.asarray(Signal_interpolated_gyroX_sp, dtype=float))
gyroY_sp = np.interp(t_sp, np.asarray(New_Time_gyroY_sp, dtype=float), np.asarray(Signal_interpolated_gyroY_sp, dtype=float))
gyroZ_sp = np.interp(t_sp, np.asarray(New_Time_gyroZ_sp, dtype=float), np.asarray(Signal_interpolated_gyroZ_sp, dtype=float))

mask_sp = (t_sp >= t_inicio_sp) & (t_sp <= t_fim_sp)

if not np.any(mask_sp):
    raise ValueError("O corte do smartphone ficou vazio (verifique t_inicio_sp/t_fim_sp).")

# Tempo absoluto e normalizado (zerado no início do corte)
tempo_sp_abs_c = t_sp[mask_sp]
tempo_sp_norm_c = tempo_sp_abs_c - t_inicio_sp

accX_sp_c = accX_sp[mask_sp]
accY_sp_c = accY_sp[mask_sp]
accZ_sp_c = accZ_sp[mask_sp]

gyroX_sp_c = gyroX_sp[mask_sp]
gyroY_sp_c = gyroY_sp[mask_sp]
gyroZ_sp_c = gyroZ_sp[mask_sp]

print(f"Smartphone | Início do corte: {t_inicio_sp:.3f} s")
print(f"Smartphone | Fim do corte:    {t_fim_sp:.3f} s  (duração = {t_fim_sp - t_inicio_sp:.3f} s)")
print(f"Smartphone | Amostras no corte: {len(tempo_sp_norm_c)}")

# DataFrame pronto para exportação (tempo normalizado + 6 sinais)
df_sp_cortado = pd.DataFrame({
    "tempo_norm_s": tempo_sp_norm_c,
    "accX_m_s2": accX_sp_c,
    "accY_m_s2": accY_sp_c,
    "accZ_m_s2": accZ_sp_c,
    "gyroX_rad_s": gyroX_sp_c,
    "gyroY_rad_s": gyroY_sp_c,
    "gyroZ_rad_s": gyroZ_sp_c,
})

Smartphone | Início do corte: 2.206 s
Smartphone | Fim do corte:    32.706 s  (duração = 30.500 s)
Smartphone | Amostras no corte: 1830


In [73]:
# =====================================================
# 10. Vicon (Esterno_Z) - Corte a partir do 1º pico (+ 30.5 s)
#     (âncora de alinhamento com o 1º ZC após o 1º pico do smartphone na exportação)
# =====================================================
from scipy.signal import find_peaks

vicon_t = np.asarray(New_Time_vc, dtype=float)
vicon_y = np.asarray(Signal_interpolated_vc, dtype=float)

# Normalizar: menor valor = 0 (mm)
vicon_y = vicon_y - np.min(vicon_y)

# Parâmetros (ajuste se necessário)
altura_minima_vc = 30.0   # mm (para picos)
duracao_corte_vc = 30.5   # s

# Picos (com limiar) e vales (todos)
picos_vc, _ = find_peaks(vicon_y, height=altura_minima_vc)
vales_vc, _ = find_peaks(-vicon_y)

if len(picos_vc) == 0:
    raise ValueError("Nenhum pico do Vicon foi detectado com o limiar atual.")

pico1_idx = int(picos_vc[0])

t_pico1_vc = float(vicon_t[pico1_idx])
t_inicio_vc = t_pico1_vc
t_fim_vc = t_pico1_vc + duracao_corte_vc

mask_corte_vc = (vicon_t >= t_inicio_vc) & (vicon_t <= t_fim_vc)

if not np.any(mask_corte_vc):
    raise ValueError("O corte temporal do Vicon ficou vazio (verifique tempos/índices).")

vicon_t_c = vicon_t[mask_corte_vc]
vicon_y_c = vicon_y[mask_corte_vc]

picos_no_corte_vc = [i for i in picos_vc if t_inicio_vc <= vicon_t[i] <= t_fim_vc]
vales_no_corte_vc = [i for i in vales_vc if t_inicio_vc <= vicon_t[i] <= t_fim_vc]

print(f"1º pico Vicon (início do corte / âncora): t = {t_pico1_vc:.3f} s (idx={pico1_idx})")
print(f"Fim do corte: t = {t_fim_vc:.3f} s  (duração = {t_fim_vc - t_inicio_vc:.3f} s)")

fig_vicon_corte = go.Figure()

fig_vicon_corte.add_trace(go.Scatter(
    x=vicon_t_c,
    y=vicon_y_c,
    mode='lines',
    name='Vicon (Esterno_Z) - Cortado',
    line=dict(color='deepskyblue')
))

if len(picos_no_corte_vc) > 0:
    fig_vicon_corte.add_trace(go.Scatter(
        x=vicon_t[picos_no_corte_vc],
        y=vicon_y[picos_no_corte_vc],
        mode='markers',
        name='Picos (no corte)',
        marker=dict(color='red', size=8, symbol='circle')
    ))

if len(vales_no_corte_vc) > 0:
    fig_vicon_corte.add_trace(go.Scatter(
        x=vicon_t[vales_no_corte_vc],
        y=vicon_y[vales_no_corte_vc],
        mode='markers',
        name='Vales (no corte)',
        marker=dict(color='blue', size=8, symbol='circle')
    ))

fig_vicon_corte.add_trace(go.Scatter(
    x=[t_pico1_vc],
    y=[vicon_y[pico1_idx]],
    mode='markers',
    name='1º pico (âncora alinhamento)',
    marker=dict(color='yellow', size=11, symbol='star')
))

fig_vicon_corte.update_layout(
    title="Vicon (Esterno_Z) — corte a partir do 1º pico (+ 30.5 s)",
    xaxis_title="Tempo (s)",
    yaxis_title="Deslocamento (mm, normalizado)",
    template="plotly_dark",
    width=1500,
    height=450,
    plot_bgcolor='black',
    paper_bgcolor='black',
    font=dict(color='white'),
    legend=dict(x=0.01, y=0.99)
)

fig_vicon_corte.show()

1º pico Vicon (início do corte / âncora): t = 1.800 s (idx=108)
Fim do corte: t = 32.300 s  (duração = 30.500 s)


In [74]:
# =====================================================
# 11. Exportar para Excel (1 arquivo): tempo normalizado + 6 smartphone + 1 vicon
# =====================================================
from scipy.interpolate import interp1d
from scipy.signal import find_peaks

# Tempo base do arquivo = tempo do smartphone no corte (absoluto)
t_sp_abs = np.asarray(tempo_sp_abs_c, dtype=float)

# Interpolar Vicon no tempo do smartphone.
# Importante: usar o Vicon COMPLETO para evitar NaN caso o corte do Vicon
# esteja em uma janela de tempo diferente do corte do smartphone.
vicon_t_full = np.asarray(New_Time_vc, dtype=float)
vicon_y_full = np.asarray(Signal_interpolated_vc, dtype=float)

# Normalizar Vicon: menor valor = 0 (mm)
vicon_y_full = vicon_y_full - np.nanmin(vicon_y_full)

def _interp_vicon(vt, vy, xq):
    f = interp1d(
        np.asarray(vt, dtype=float),
        np.asarray(vy, dtype=float),
        kind="linear",
        bounds_error=False,
        fill_value=np.nan,
    )
    return f(np.asarray(xq, dtype=float))

# Alinhamento: 1º ZC após o 1º pico (smartphone) = 1º pico (Vicon)
# Mesmo instante físico: t_vc = t_sp + (t_pico1_vc - t_zc_apos_pico_sp)
align_shift = float(t_pico1_vc - t_zc_apos_pico_sp)
peak_extra_sp = 0.0  # correção do mapeamento t_vc→t_sp se usar fallback abaixo
viconZ_on_sp = _interp_vicon(vicon_t_full, vicon_y_full, t_sp_abs + align_shift)
print(f"Alinhamento Vicon↔telefone | shift na interpolação: {align_shift:.3f} s")

# Se ficar majoritariamente NaN, segunda tentativa: corrigir relógio entre inícios de corte
if np.mean(np.isnan(viconZ_on_sp)) > 0.5:
    offset_inicio = float(t_inicio_vc - t_inicio_sp)
    t_query = t_sp_abs + align_shift - offset_inicio
    viconZ_on_sp = _interp_vicon(vicon_t_full, vicon_y_full, t_query)
    peak_extra_sp = offset_inicio
    print(f"[aviso] Vicon fora da janela; amostragem com correção de relógio {offset_inicio:.3f} s")

# --- Gráfico de verificação do alinhamento (antes do Excel) ---
t_norm = np.asarray(df_sp_cortado["tempo_norm_s"], dtype=float)
x_alinha = float(t_zc_apos_pico_sp - t_inicio_sp)

# Mesmo recorte do smartphone, mas acel. Y filtrada (fc = 1 Hz), como na detecção de picos
_t_sp_axis = np.asarray(New_Time_accY_sp, dtype=float)
_mask_sp_plot = (_t_sp_axis >= float(t_inicio_sp)) & (_t_sp_axis <= float(t_fim_sp))
accY_filt_cortado = np.asarray(Signal_filtered_accY_sp, dtype=float)[_mask_sp_plot]
if accY_filt_cortado.shape[0] != t_norm.shape[0]:
    raise ValueError("Comprimento accY filtrada ≠ tempo normalizado no gráfico de alinhamento.")

fig_alinha = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("Smartphone — acel. Y filtrada (fc=1 Hz, cortado)", "Vicon Esterno_Z (interp. alinhada)"),
)
fig_alinha.add_trace(
    go.Scatter(x=t_norm, y=accY_filt_cortado, name="accY filtrada", line=dict(color="deepskyblue")),
    row=1, col=1,
)
fig_alinha.add_trace(
    go.Scatter(x=t_norm, y=viconZ_on_sp, name="Vicon Z", line=dict(color="orange")),
    row=2, col=1,
)

# Picos do Vicon (mesmo limiar do bloco Vicon) → tempo normalizado no eixo do telefone
_alt_vc = 30.0
_picos_vc_idx, _ = find_peaks(np.asarray(vicon_y_full, dtype=float), height=_alt_vc)
_t0_sp, _tf_sp = float(t_inicio_sp), float(t_fim_sp)
_xmax = float(t_norm[-1]) if t_norm.size else 0.0

def _xnorm_from_tvc(t_vc):
    return float(t_vc) - align_shift + peak_extra_sp - _t0_sp

for _pi in _picos_vc_idx:
    _xn = _xnorm_from_tvc(vicon_t_full[_pi])
    if -0.05 <= _xn <= _xmax + 0.05:
        for _ri in (1, 2):
            fig_alinha.add_vline(
                x=_xn, line_color="magenta", line_width=1, opacity=0.85,
                row=_ri, col=1,
            )

# Zero crossings da acel. Y filtrada (eixo completo), projetados no tempo normalizado do corte
_sfil = np.asarray(Signal_filtered_accY_sp, dtype=float)
_tfil = np.asarray(New_Time_accY_sp, dtype=float)
_zci = np.where(np.diff(np.sign(_sfil)))[0]
_zc_t_list = []
for _i in _zci:
    _x1, _x2 = _tfil[_i], _tfil[_i + 1]
    _y1, _y2 = _sfil[_i], _sfil[_i + 1]
    if (_y2 - _y1) != 0:
        _zc_t_list.append(float(_x1 - _y1 * (_x2 - _x1) / (_y2 - _y1)))
for _tzc in _zc_t_list:
    if _t0_sp <= _tzc <= _tf_sp:
        _xn = _tzc - _t0_sp
        for _ri in (1, 2):
            fig_alinha.add_vline(
                x=_xn, line_dash="dot", line_color="cyan", line_width=1, opacity=0.75,
                row=_ri, col=1,
            )

for ri in (1, 2):
    fig_alinha.add_vline(
        x=x_alinha, line_dash="dash", line_color="lime", line_width=2,
        row=ri, col=1,
    )
fig_alinha.add_trace(
    go.Scatter(x=[None], y=[None], mode="lines", name="Picos Vicon (vertical)", line=dict(color="magenta", width=2)),
    row=1, col=1,
)
fig_alinha.add_trace(
    go.Scatter(x=[None], y=[None], mode="lines", name="ZC smartphone Y filt. (vertical)", line=dict(color="cyan", width=2, dash="dot")),
    row=1, col=1,
)
fig_alinha.add_annotation(
    x=x_alinha, yref="paper", y=1.12, showarrow=False,
    text="1º ZC após pico (telefone) = 1º pico (Vicon)",
    font=dict(color="white", size=12),
)
fig_alinha.update_layout(
    title="Verificação: picos Vicon (magenta) vs ZC telefone (cyan); âncora (verde); acel. Y filtrada",
    template="plotly_dark", height=700, width=1500,
    plot_bgcolor="black", paper_bgcolor="black", font=dict(color="white"),
)
fig_alinha.update_xaxes(title_text="Tempo normalizado (s) desde o ZC antes do 1º pico", row=2, col=1)
fig_alinha.show()

df_export = df_sp_cortado.copy()
df_export["vicon_esternoZ_mm_norm"] = viconZ_on_sp

# Diagnóstico rápido
nan_ratio = float(np.mean(np.isnan(df_export["vicon_esternoZ_mm_norm"].to_numpy())))
print(f"Vicon no Excel | NaN ratio: {nan_ratio:.3f}")

# Salvar
out_dir = "Output_ML"
os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(out_dir, f"{sujeito}_smartphone_vicon_cortado.xlsx")

df_export.to_excel(out_path, index=False)

print(f"Arquivo salvo em: {out_path}")
print(f"Linhas: {len(df_export)} | Colunas: {len(df_export.columns)}")

Alinhamento Vicon↔telefone | shift na interpolação: -0.659 s


Vicon no Excel | NaN ratio: 0.000
Arquivo salvo em: Output_ML/03_smartphone_vicon_cortado.xlsx
Linhas: 1830 | Colunas: 8
